In [64]:
import json
import os
import hashlib

from rockyfilescom.wrapper import Session as RFC
from rockyclickup.wrapper import Session as RCU
from rockyclickup.utils import response_to_model
from rockyclickup.models import DataFile


from odin import PostgresWrapper

In [21]:
INSTANCE = os.getenv("DATAINBOX_DB_CONNECTION_NAME")
DB_NAME = os.getenv("DATAINBOX_DB_NAME")
USER = os.getenv("DATAINBOX_USER")
PASSWORD = os.getenv("DATAINBOX_PASSWORD")
IP_TYPE = os.getenv("DATAINBOX_DB_IP_TYPE", "public")

In [34]:
rfc = RFC()
rcu = RCU()

In [23]:
datainbox_db = PostgresWrapper(
    instance_connection_name = INSTANCE,
    db_name = DB_NAME,
    user = USER,
    password = PASSWORD,
    ip_type = IP_TYPE,
)

In [36]:
with open("../../saved_message_data.json", "r") as f:
    message_data = json.load(f)

message_data

{'default': {'source': 'Files.com'},
 'action': 'create',
 'interface': 'desktop',
 'path': 'Clients/Orange Tree Co-RMROTREE/File Feeds/webhook.txt',
 'at': '2026-08-12T11:43:49-04:00',
 'username': 'james.richmond@rmrbenefits.com',
 'ip': '73.20.57.233',
 'type': 'file',
 'size': 177572}

In [37]:
filescom_file = rfc.get_file(path=message_data['path'])

filescom_file

In [85]:
# task_id = filescom_file.custom_metadata.get("task_id", None)


# '''steps 1-5 in handlers.py reconcile()'''
# if not task_id:
#     # need to search clickup for datafile by path and name
#     # if found:
#         # backfill files.com custom metadata

#     # else:
#         # create datafile on clickup
#         # backfill files.com custom metadata
    
#     print(f"files.com custom_metadata did not have clickup task_id")



# # get the task from clickup
# raw_json = rcu.get_task_by_id(task_id)
# # model = response_to_model(raw_json)

def narrow_task(raw_task: dict):
    model = response_to_model(raw_task)

    return {
        "task_id": model.id,
        "task_name": model.name,
        "status": model.status,
        "archived": model.archived,
        "assignees": model.assignees,
        "file_name": model.ftp_filename,
        "file_directory": model.ftp_directory,
        "category": model.file_category,
        "date_received": model.received,
        "date_task_created": raw_task.get("date_created"),
        "date_task_updated": raw_task.get("date_updated"),
        # "date_created": model.date_created,
        # "date_modified": model.date_modified,
        "datacard": model.inbox,
        # "checksum": model.checksum,
        "raw_json": raw_task
    }


def generate_checksum(narrowed_task):
    exclude = {"raw_json", "date_received", "date_task_created", "date_task_updated"}
    dict_to_encode = {k: v for k, v in narrowed_task.items() if k not in exclude}

    encoded = json.dumps(dict_to_encode, sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()




In [87]:

def sync_database(task_id: str):

    '''step 10 in handlers.py reconcile()'''

    # get task from clickup
    raw_json = rcu.get_task_by_id(task_id)

    # check to see if task id appears in database
    exists = datainbox_db.get_table_item_by_attribute("datafiles", "task_id", task_id)

    # if not found, create in database
    if exists is None:
        narrow = narrow_task(raw_json)
        checksum = generate_checksum(narrow)

        narrow["checksum"] = checksum
        narrow["raw_json"] = raw_json

        datainbox_db.add_item_to_table("datafiles", values=narrow)
        print("task has been created in database")

    else:
        # check if we need to update in the database
        print("need to update in database")

        # check if clickups 'date_updated' matches what we have recorded in the database
        task_last_update = raw_json.get("date_updated")
        row_last_update = exists.get("date_task_updated")

        # if they match, we dont need to update anything
        if task_last_update == row_last_update:
            print("task is already up to date!")
            return

        # check if new checksum matches what we have recorded in the database
        narrow = narrow_task(raw_json)
        new_checksum = generate_checksum(narrow)

        narrow["checksum"] = new_checksum
        narrow["raw_json"] = raw_json

        # create update object and send changes to database
        datainbox_db.update_table_item("datafiles", values=narrow, where={"task_id": task_id})
        if exists.get("checksum") == new_checksum:
            print("task updated (raw_json/backup refreshed only — no narrow field changes)")
        else:
            print("task updated (narrow fields changed)")


In [88]:
sync_database("868kqh5yd")

need to update in database
task updated (narrow fields changed)


In [69]:
"868kqh5yd"

'868kqh5yd'

In [91]:
row = datainbox_db.get_table_item_by_attribute("datafiles", "task_id", "868kqh5yd")

In [92]:
row.get("raw_json").get("tags")

[{'name': 'debug',
  'tag_bg': '#ff5251',
  'tag_fg': '#ff5251',
  'creator': 50835299}]